# Práctico 3 – Tu primer agente con tools

**Magíster en Inteligencia Artificial UC · INF3880 – Inteligencia Artificial Agéntica**

**Profesores:** Vicente Domínguez, Antonio Ossa · **Ayudante:** Michelle Madrid

**Nombre del alumno:** ESCRIBIR NOMBRE AQUÍ

> Este práctico **no es evaluado**. Son ejercicios propuestos para trabajar en el Módulo 2 de la clase 3 (lunes 02/11/2026) y terminar en casa si hace falta. La versión con soluciones se publica después de la clase.

Construiremos el **loop agéntico a mano** con *function calling* de Gemini, para entender cada paso, y luego dejaremos que el SDK haga el loop por nosotros. El caso: un asistente interno de la empresa de logística *Andina Cargo*, que reutilizaremos en varios prácticos. Seguimos con la **API gratuita de Gemini** y tu misma key.


## Configuración inicial

Usaremos la **API de Gemini en su nivel gratuito** (no requiere tarjeta de crédito). Cada alumno crea su propia key; toma unos 2 minutos.

### Paso A · Crear tu API key en Google AI Studio
1. Abre **https://aistudio.google.com/apikey** e inicia sesión con tu cuenta de Google (personal; la cuenta institucional puede tener AI Studio bloqueado).
2. La primera vez te pedirá **aceptar los términos de servicio** de la Gemini API. Al aceptarlos, Google crea automáticamente un proyecto por defecto.
3. Haz clic en **"Create API key"** (si ya tienes un proyecto, elige "Create API key in existing project" o el proyecto por defecto).
4. Copia la key con el botón **"Copiar clave"** (las keys actuales empiezan con `AQ.`; las antiguas con `AIza`). Guárdala en un lugar seguro: la vas a necesitar en el paso B. Si la pierdes, puedes crear otra desde la misma página.
5. Opcional pero recomendado: en la lista de keys, pasa el mouse sobre la etiqueta **"Unrestricted"** → **"Add restrictions"** → **"Restrict to Gemini API only"** → **"Restrict key"**.

> ¿Dónde ver los límites y el uso? En AI Studio, menú lateral → **Dashboard** → **Usage & Billing** (o https://aistudio.google.com/usage). Ahí aparece el plan "Free" y las cuotas por modelo.
>
> ¿Problemas? Si el botón "Create API key" no aparece, suele ser porque la cuenta es de una organización (Workspace) que restringe AI Studio: usa una cuenta personal de Gmail. Documentación oficial: https://ai.google.dev/gemini-api/docs/api-key

### Paso B · Guardar la key en Secrets de Colab
1. En este notebook, haz clic en el ícono de la **llave 🔑** de la barra lateral izquierda ("Secrets").
2. **"+ Add new secret"**: en *Name* escribe exactamente `GEMINI_API_KEY`; en *Value* pega la key.
3. Activa el interruptor **"Notebook access"** para este notebook.
4. **Nunca pegues la key en una celda, ni la subas a un repositorio, ni la compartas.** Trátala como una contraseña: si se filtra, crea una nueva y borra la antigua en AI Studio.

### Paso C · Instalar y configurar
Instalamos la librería `google-genai` y definimos el modelo en una sola variable `MODEL` para poder cambiarlo sin tocar el resto del notebook.

> ⚠️ **Dos cosas que debes saber del nivel gratuito.** (1) Según los términos de Google, los prompts y respuestas del nivel sin pago **pueden usarse para mejorar sus productos y ser revisados por personas**: no envíes datos personales, confidenciales ni de tu empresa. (2) Tiene **límites de uso por minuto y por día**; si recibes un error 429 (*resource exhausted*), espera un momento y reintenta. Los límites de tu proyecto se ven en AI Studio.

In [ ]:
%%capture
!pip install -q --upgrade google-genai pandas pydantic

In [ ]:
import os, time
from google.colab import userdata

# 1) Leer la key desde Secrets, con mensajes claros si algo falta
try:
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except userdata.SecretNotFoundError:
    raise SystemExit("No existe un secreto llamado GEMINI_API_KEY. Créalo en el panel de Secrets (ícono de la llave) con ese nombre exacto.")
except userdata.NotebookAccessError:
    raise SystemExit("El secreto GEMINI_API_KEY existe pero no tiene 'Notebook access' activado para este notebook.")
assert GEMINI_API_KEY and len(GEMINI_API_KEY) > 20, \
    "El secreto GEMINI_API_KEY está vacío. Pega la key completa desde https://aistudio.google.com/apikey (botón 'Copiar clave')."
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

# 2) Elegir el modelo: buscamos uno que esté disponible en TU proyecto gratuito y que respete `temperature`
#    (los modelos Gemini 3.6+ y 4 la ignoran, por eso preferimos Flash-Lite 3.5 o 2.5)
from google import genai
client = genai.Client()
PREFERIDOS = ["gemini-3.5-flash-lite", "gemini-2.5-flash-lite", "gemini-3.1-flash-lite", "gemini-2.5-flash", "gemini-3.8-flash"]
disponibles = []
for m in client.models.list():
    acciones = getattr(m, "supported_actions", None) or []
    if not acciones or "generateContent" in acciones:
        disponibles.append(m.name.replace("models/", ""))
MODEL = next((p for p in PREFERIDOS if p in disponibles), None)
if MODEL is None:
    print("Ninguno de los modelos preferidos está disponible. Modelos de tu proyecto:", disponibles[:30])
    MODEL = next((d for d in disponibles if "flash" in d), disponibles[0])
RESPALDO = [p for p in PREFERIDOS if p in disponibles and p != MODEL]   # por si el modelo elegido falla (error 500/503)
print("Modelo elegido:", MODEL, "| respaldo:", RESPALDO)

# Precios de referencia del nivel DE PAGO, en USD por 1M tokens (https://ai.google.dev/gemini-api/docs/pricing).
# En el nivel gratuito no pagas, pero calcularemos cuánto costaría para entender la economía de un LLM.
PRECIOS = {
    "gemini-3.5-flash-lite": {"input": 0.30, "output": 2.50},
    "gemini-3.8-flash":      {"input": 0.75, "output": 3.75},
    "gemini-3.1-flash-lite": {"input": 0.25, "output": 1.50},
    "gemini-2.5-flash":      {"input": 0.30, "output": 2.50},
    "gemini-2.5-flash-lite": {"input": 0.10, "output": 0.40},
}

def costo_usd(input_tokens, output_tokens, model=None):
    p = PRECIOS.get(model or MODEL, PRECIOS["gemini-3.5-flash-lite"])   # si el modelo no está en la tabla, usa un precio de referencia
    return ((input_tokens or 0) * p["input"] + (output_tokens or 0) * p["output"]) / 1_000_000

print("Listo. Modelo:", MODEL)

Verificamos que la key funciona:

In [ ]:
from google.genai import types

def generar(contents, reintentos=3, **config):
    """Llama al modelo con manejo de los errores típicos del nivel gratuito:
    - 429 (límite de uso): espera y reintenta.
    - 500/503 (servidor saturado o modelo no disponible en ese momento): reintenta con esperas crecientes
      y, si sigue fallando, cambia al siguiente modelo de RESPALDO."""
    global MODEL
    for intento in range(reintentos + 1):
        try:
            return client.models.generate_content(model=MODEL, contents=contents,
                                                  config=types.GenerateContentConfig(**config))
        except Exception as e:
            msg = str(e)
            if ("429" in msg or "RESOURCE_EXHAUSTED" in msg) and intento < reintentos:
                print("  (límite de uso alcanzado, esperando 20 s...)"); time.sleep(20)
            elif any(k in msg for k in ("500", "503", "INTERNAL", "UNAVAILABLE", "overloaded", "not available")):
                if intento < reintentos:
                    espera = 5 * 2 ** intento          # 5, 10, 20 s
                    print(f"  (error transitorio del servidor, reintentando en {espera} s...)"); time.sleep(espera)
                elif RESPALDO:
                    anterior, MODEL = MODEL, RESPALDO.pop(0)
                    print(f"  ⚠️ {anterior} sigue fallando; cambiamos a {MODEL} y seguimos.")
                    return generar(contents, reintentos=reintentos, **config)
                else:
                    raise SystemExit("Gemini devolvió errores 500/503 en todos los modelos disponibles. Suele ser saturación temporal: espera unos minutos y vuelve a ejecutar esta celda.")
            elif "404" in msg or "NOT_FOUND" in msg:
                raise SystemExit(f"El modelo {MODEL} no está disponible en tu proyecto. Cambia MODEL por otro de la lista de la celda anterior.")
            elif "403" in msg or "401" in msg or "API_KEY_INVALID" in msg:
                raise SystemExit("La API key fue rechazada. Verifica que la copiaste completa desde https://aistudio.google.com/apikey y que no está restringida a otra API.")
            else:
                raise

print(generar("Responde solo: OK").text)

In [ ]:
PAUSA = 2.0   # segundos entre llamadas al modelo, para respetar el límite por minuto del nivel gratuito

## Parte guiada 1 · Los datos de Andina Cargo

A partir de este práctico trabajaremos con un **asistente interno de una empresa de logística**, *Andina Cargo*. Creamos datos de ejemplo (pedidos, stock, tiempos de despacho) que simulan sus sistemas internos.

In [ ]:
import json, pathlib

PEDIDOS = [
  {"id": "P-1001", "cliente": "Ferretería Lagos", "ciudad": "Valparaíso", "estado": "en_transito", "items": [{"sku": "TAL-200", "cant": 10}], "fecha_despacho": "2026-10-15", "eta_dias": 2},
  {"id": "P-1002", "cliente": "Minimarket Sol", "ciudad": "Santiago", "estado": "entregado", "items": [{"sku": "CAJ-050", "cant": 200}], "fecha_despacho": "2026-10-10", "eta_dias": 0},
  {"id": "P-1003", "cliente": "Clínica Norte", "ciudad": "Antofagasta", "estado": "preparacion", "items": [{"sku": "GUA-010", "cant": 50}, {"sku": "TAL-200", "cant": 5}], "fecha_despacho": None, "eta_dias": 4},
  {"id": "P-1004", "cliente": "Ferretería Lagos", "ciudad": "Valparaíso", "estado": "retrasado", "items": [{"sku": "CAJ-050", "cant": 40}], "fecha_despacho": "2026-10-12", "eta_dias": 3},
  {"id": "P-1005", "cliente": "Agro Sur", "ciudad": "Temuco", "estado": "en_transito", "items": [{"sku": "PAL-001", "cant": 12}], "fecha_despacho": "2026-10-16", "eta_dias": 3},
]
STOCK = {"TAL-200": {"nombre": "Taladro 200W", "stock": 35, "bodega": "Santiago"},
         "CAJ-050": {"nombre": "Caja cartón 50x50", "stock": 0, "bodega": "Santiago"},
         "GUA-010": {"nombre": "Guantes nitrilo (caja 100)", "stock": 120, "bodega": "Antofagasta"},
         "PAL-001": {"nombre": "Pallet madera", "stock": 8, "bodega": "Temuco"}}
DIAS_DESPACHO_ESTIMADOS = {"Santiago": 1, "Valparaíso": 2, "Temuco": 3, "Antofagasta": 4, "Punta Arenas": 7}

pathlib.Path("data").mkdir(exist_ok=True)
json.dump(PEDIDOS, open("data/pedidos.json", "w"), ensure_ascii=False, indent=2)
json.dump(STOCK, open("data/stock.json", "w"), ensure_ascii=False, indent=2)
json.dump(DIAS_DESPACHO_ESTIMADOS, open("data/dias_despacho.json", "w"), ensure_ascii=False, indent=2)
print(f"{len(PEDIDOS)} pedidos, {len(STOCK)} SKUs y {len(DIAS_DESPACHO_ESTIMADOS)} lugares de despacho guardados en data/")

## Parte guiada 2 · Las tools: funciones Python + un contrato para el modelo

Un LLM solo produce texto: no puede leer tu base de datos ni ejecutar nada. Una **tool** es una función tuya que el modelo puede *pedir* que ejecutes. Para eso necesita dos cosas:

1. La **función Python** de siempre (acá consulta listas en memoria; en la vida real sería una API o una base de datos).
2. Una **declaración**: nombre, descripción y parámetros en *JSON Schema*. Es lo único que el modelo ve; la descripción funciona como "el prompt de la tool".

In [ ]:
import json, time
from google.genai import types


# --- 1) Las funciones Python: código normal, sin nada de IA. Devuelven dict (o {"error": ...}) ---
def buscar_pedido(id_pedido: str) -> dict:
    """Devuelve el pedido con ese id: estado, cliente, ciudad, items, fecha de despacho y ETA en días."""
    try:
        with open("data/pedidos.json", "r", encoding="utf-8") as f:
            pedidos_db = json.load(f)
    except Exception as e:
        return {"error": f"Error al leer la base de datos de pedidos: {str(e)}"}

    p = next((p for p in pedidos_db if p["id"] == id_pedido.upper().strip()), None)
    return p if p else {"error": f"No existe el pedido {id_pedido}"}

def consultar_stock(sku: str) -> dict:
    """Devuelve nombre, unidades disponibles y bodega de un SKU."""
    try:
        with open("data/stock.json", "r", encoding="utf-8") as f:
            stock_db = json.load(f)
    except Exception as e:
        return {"error": f"Error al leer el inventario: {str(e)}"}

    s = stock_db.get(sku.upper().strip())
    return {"sku": sku.upper().strip(), **s} if s else {"error": f"SKU {sku} no encontrado"}

def estimar_despacho(ciudad: str) -> dict:
    """Días hábiles estimados de despacho a una ciudad chilena."""
    try:
        with open("data/dias_despacho.json", "r", encoding="utf-8") as f:
            despachos_db = json.load(f)
    except Exception as e:
        return {"error": f"Error al leer la tabla de despachos: {str(e)}"}

    dias = despachos_db.get(ciudad.strip().title())
    if dias is None:
        return {"error": f"Sin cobertura registrada para {ciudad}", "ciudades_con_cobertura": list(despachos_db)}
    return {"ciudad": ciudad.strip().title(), "dias_habiles": dias}

printf("Buscar Pedido ID: P-1004")
printf(buscar_pedido("P-1004"))

printf("Consultar Stock para Producto SKU: CAJ-050")
printf(consultar_stock("CAJ-050"))

printf("Estimar Despacho para Temuco")
printf(estimar_despacho("Temuco"))

In [ ]:
# --- 2) El "contrato" que ve el modelo: nombre, descripción y parámetros (JSON Schema). El modelo NUNCA ve el código. ---
DECL_BUSCAR_PEDIDO = types.FunctionDeclaration(
    name="buscar_pedido",
    description="Busca un pedido por su id (formato P-1234) y devuelve estado, cliente, ciudad, items, fecha de despacho y ETA en días.",
    parameters_json_schema={
        "type": "object",
        "properties": {"id_pedido": {"type": "string", "description": "Id del pedido, por ejemplo P-1001"}},
        "required": ["id_pedido"],
    },
)
DECL_CONSULTAR_STOCK = types.FunctionDeclaration(
    name="consultar_stock",
    description="Consulta las unidades disponibles de un producto por su SKU (formato ABC-123) y en qué bodega está.",
    parameters_json_schema={
        "type": "object",
        "properties": {"sku": {"type": "string", "description": "Código SKU, por ejemplo TAL-200"}},
        "required": ["sku"],
    },
)

TOOLS = [types.Tool(function_declarations=[DECL_BUSCAR_PEDIDO, DECL_CONSULTAR_STOCK])]
FUNCIONES = {"buscar_pedido": buscar_pedido, "consultar_stock": consultar_stock}   # nombre -> función real
print("Tools declaradas:", list(FUNCIONES))

💡 **Observa:** declaramos solo dos de las tres funciones. Para el modelo, `estimar_despacho` **no existe** todavía: lo comprobaremos más abajo. Una tool mal descrita o no declarada simplemente no se usa.

## Parte guiada 3 · El loop agéntico a mano

Ahora escribimos el ciclo completo, sin que el SDK haga nada por nosotros, para entender cada paso:

1. Enviamos la pregunta junto con las declaraciones de tools (`tools=TOOLS`) y desactivamos la ejecución automática del SDK.
2. Si la respuesta trae `function_calls`, **nosotros** ejecutamos la función en Python.
3. Devolvemos el resultado como `function_response` y volvemos a llamar al modelo con todo el historial.
4. Repetimos hasta que responda con texto o lleguemos a un límite de iteraciones.

Primero, el system prompt (fíjate en las reglas: usar siempre las tools, pedir datos faltantes, explicar errores).

In [ ]:
INSTRUCCIONES = """Eres el asistente interno de Andina Cargo, una empresa de logística chilena.
Respondes consultas de ejecutivos de atención a clientes sobre pedidos, stock y tiempos de despacho.

Reglas:
- Usa SIEMPRE las herramientas para obtener datos; nunca inventes ids, estados ni cantidades.
- Si falta información (por ejemplo, el id del pedido), pídela antes de responder.
- Si una herramienta devuelve un error, explícalo con claridad y propone qué hacer.
- Responde en español, en máximo 4 líneas, con tono profesional y directo.
"""

In [ ]:
USO = []   # acumulamos usage_metadata de cada llamada para medir cuánto cuesta un loop

def agente_manual(pregunta, max_iter=5, verbose=True, tools=None, funciones=None):
    """Loop agéntico mínimo: decidir -> actuar -> observar, hasta que haya respuesta final o se agote max_iter."""
    tools, funciones = tools or TOOLS, funciones or FUNCIONES
    contenido = [types.Content(role="user", parts=[types.Part.from_text(text=pregunta)])]
    for paso in range(1, max_iter + 1):
        r = generar(contenido, system_instruction=INSTRUCCIONES, tools=tools, temperature=0,
                    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True))
        USO.append(r.usage_metadata)
        contenido.append(r.candidates[0].content)          # guardamos el turno del modelo (texto o pedido de tool)
        if not r.function_calls:                             # no pidió ninguna tool: es la respuesta final
            return r.text
        partes = []
        for call in r.function_calls:                        # puede pedir varias tools en la misma vuelta
            args = dict(call.args)
            resultado = funciones[call.name](**args)         # NOSOTROS ejecutamos la función, no el modelo
            if verbose: print(f"  paso {paso}: {call.name}({args}) -> {resultado}")
            partes.append(types.Part.from_function_response(name=call.name, response={"resultado": resultado}))
        contenido.append(types.Content(role="user", parts=partes))   # le devolvemos los resultados (notar que usamos "user")
        time.sleep(PAUSA)
    return "Se alcanzó el límite de iteraciones sin respuesta final."

print(agente_manual("¿En qué estado está el pedido P-1004 y hay stock para reponer lo que lleva?"))

💡 **Observa:** el modelo **no ejecutó nada**: pidió `buscar_pedido`, nosotros lo ejecutamos, le devolvimos el resultado, y con eso decidió pedir `consultar_stock`. Ese ciclo *decidir → actuar → observar* es el loop agéntico; todo lo demás (frameworks, memoria, multiagentes) se construye encima. Cada vuelta es una llamada al modelo: cuenta para el límite por minuto del nivel gratuito.

In [ ]:
# ¿Y si preguntamos por algo que requiere una tool NO declarada?
print(agente_manual("¿Cuántos días hábiles tarda un despacho a Temuco?"))

💡 **Observa:** sin la declaración, el modelo no puede usar `estimar_despacho`. Lo esperable, por la regla 1 del system prompt, es que diga que no tiene esa información o la pida; si en cambio **inventa** un número, acabas de ver por qué las reglas del prompt y las tools se diseñan juntas. En el ejercicio 1 vas a declarar una tool nueva.

In [ ]:
# ¿Cuánto costó el loop? Cada vuelta reenvía TODO el historial, así que el costo crece con los pasos.
entrada = sum((u.prompt_token_count or 0) for u in USO)
salida = sum((u.candidates_token_count or 0) for u in USO)
print(f"{len(USO)} llamadas al modelo | tokens entrada: {entrada} | salida: {salida} | costo ≈ US${costo_usd(entrada, salida):.5f}")

## Parte guiada 4 · El SDK hace el loop por ti (function calling automático)

Lo que escribimos a mano es tan común que el SDK de Gemini lo trae incorporado: si le pasas **funciones Python directamente** en `tools`, genera la declaración a partir de los *type hints* y el *docstring*, ejecuta las llamadas y repite hasta la respuesta final. Esto es, en esencia, lo que hacen los frameworks de agentes que veremos desde la clase 5.

In [ ]:
r = generar("¿Cuándo le llega a Clínica Norte su pedido P-1003?",
            system_instruction=INSTRUCCIONES, temperature=0,
            tools=[buscar_pedido, consultar_stock, estimar_despacho])   # funciones Python, sin declaración manual
print(r.text)

In [ ]:
# El "trace": qué pasó en cada turno del loop automático
for turno in (r.automatic_function_calling_history or []):
    for parte in (turno.parts or []):
        if parte.function_call:
            print(f"[{turno.role}] pide {parte.function_call.name}({dict(parte.function_call.args)})")
        elif parte.function_response:
            print(f"[{turno.role}] resultado -> {parte.function_response.response}")
        elif parte.text:
            print(f"[{turno.role}] {parte.text[:120]}")

💡 **Observa:** el historial automático es exactamente la secuencia que construimos a mano: turno del usuario, pedido de tool, resultado, (quizás otro pedido), respuesta. Cuando un framework "hace magia", esto es lo que está pasando por debajo. Saber escribirlo a mano te permite depurarlo cuando falla.

### (¿Y ese *warning*?)

El SDK de Gemini maneja las conversaciones mediante `client.chats.create`. Al crear el chat, le pasamos la configuración del sistema (`system_instruction`) y las herramientas disponibles (`tools`). Luego, cada interacción se realiza de forma limpia mediante chat.`send_message(...)`, permitiendo que la librería gestione automáticamente todo el ciclo agéntico por debajo de forma óptima.

```python
# Creamos la configuración del chat incluyendo las herramientas y la instrucción de sistema
config_chat = types.GenerateContentConfig(
    system_instruction=INSTRUCCIONES,
    temperature=0,
    tools=[buscar_pedido, consultar_stock, estimar_despacho]
)

# Iniciamos un nuevo chat con la configuración establecida
chat = client.chats.create(model=MODEL, config=config_chat)

# Enviamos el mensaje al chat. El SDK resolverá las llamadas de función automáticamente sin arrojar warnings
respuesta = chat.send_message("¿Cuándo le llega a Clínica Norte su pedido P-1003?")

print(respuesta.text)
```


¿Por qué no usamos esto en el curso?

1. **Entender el ciclo de vida "a mano"**: Para entender cómo funciona la IA agéntica, el objetivo de este paso es ver lo que ocurre en cada paso. Usando la API de chat, el SDK oculta los turnos intermedios que es lo que queremos ver.

2. **Control sobre las respuestas en el nivel gratuito**: La función `generar` incluye manejo de errores 429 (límite de cuota) y 500/503 (caída del modelo), usando errores legibles y amigables.


## Ejercicios propuestos

De dificultad creciente. El último siempre es abierto. Si te trabas, revisa las pistas y la parte guiada; la solución se publica después de la clase.

### Ejercicio 1 – Una tercera tool declarada a mano

Implementa `fecha_estimada_entrega(id_pedido)`, que devuelva la fecha estimada de entrega (`fecha_despacho` + `eta_dias`) o un mensaje claro si el pedido aún no fue despachado. Escribe su `FunctionDeclaration`, agrégala a `TOOLS` y `FUNCIONES`, y prueba `agente_manual` con los pedidos **P-1001** (despachado) y **P-1003** (sin despachar). ¿La descripción fue suficiente para que el modelo eligiera la tool nueva en vez de `buscar_pedido`?

**Pistas:** `date.fromisoformat(p['fecha_despacho']) + timedelta(days=p['eta_dias'])`. Reutiliza `buscar_pedido` dentro de la función nueva. Para el modelo, lo que distingue una tool de otra es la **descripción**: di explícitamente cuándo usarla.

In [ ]:
from datetime import date, timedelta

def fecha_estimada_entrega(id_pedido: str) -> dict:
    """Devuelve la fecha estimada de entrega (fecha_despacho + eta_dias) o un mensaje si aún no se despachó."""
    # TODO
    ...

DECL_FECHA = types.FunctionDeclaration(
    name="fecha_estimada_entrega",
    description="TODO",
    parameters_json_schema={"type": "object", "properties": {}, "required": []},   # TODO
)

# TODO: crear TOOLS y FUNCIONES ampliados y probar con P-1001 y P-1003


### Ejercicio 2 – Límite de iteraciones

Un loop sin tope puede quedarse pidiendo tools para siempre (y cobrándote cada vuelta).

- a) Ejecuta `agente_manual` con la pregunta encadenada de la parte guiada 3 y `max_iter=1`: ¿qué devuelve?

- b) En la variante automática, limita el loop con `AutomaticFunctionCallingConfig(maximum_remote_calls=1)` y observa qué trae la respuesta (`r.text`, `r.function_calls`). Luego responde: en un sistema real, ¿qué debería hacer la aplicación cuando el agente se corta por límite?

**Pistas:** En (a) `agente_manual` ya tiene el parámetro. En (b) `r.text` puede ser `None`.

In [ ]:
# TODO (a): agente_manual(..., max_iter=1)


In [ ]:
# TODO (b): generar(..., tools=[buscar_pedido, consultar_stock],
#                    automatic_function_calling=types.AutomaticFunctionCallingConfig(maximum_remote_calls=1))

### Ejercicio 3 – Provocar y manejar un error de tool

Las tools fallan: APIs caídas, timeouts, datos corruptos. Crea `consultar_stock_roto`, que lance `RuntimeError('Inventario no disponible')`, y (1) ejecuta `agente_manual` con un diccionario de funciones donde `consultar_stock` apunte a la versión rota: observa el *traceback* (el loop muere). (2) Escribe `agente_robusto`, una copia del loop que capture la excepción y le devuelva al modelo `{"error": "..."}` como resultado de la tool. Comprueba que el modelo **explica el problema** en vez de inventar un stock (regla 3 del system prompt).

**Pistas:** Un `try/except Exception as e` alrededor de `funciones[call.name](**args)`; el mensaje de error va en el mismo lugar donde iría el resultado. Compara la respuesta con lo que pide la regla 3 del system prompt.

In [ ]:
def consultar_stock_roto(sku: str) -> dict:
    """Simula que el sistema de inventario está caído."""
    raise RuntimeError("Inventario no disponible (timeout 30 s)")

FUNCIONES_ROTAS = {"buscar_pedido": buscar_pedido, "consultar_stock": consultar_stock_roto}

# TODO (1): agente_manual(PREGUNTA, funciones=FUNCIONES_ROTAS)  -> observa el traceback

# TODO (2): def agente_robusto(...): copia de agente_manual con try/except alrededor de la ejecución de la tool


### Ejercicio 4 – Abierto: tools para tu trabajo

Diseña 3 tools para un asistente de tu área (nombre, descripción, parámetros) e implementa versiones *mock* que devuelvan datos de ejemplo. Pásalas como funciones Python a `generar` (parte guiada 4) y pruébalas con 3 preguntas, una de las cuales debe requerir **encadenar dos tools**. ¿Qué descripción tuviste que mejorar para que el modelo eligiera bien?

In [ ]:
# TODO


## Para seguir en casa

- Tu key gratuita de Gemini sigue funcionando fuera de clase: los límites diarios se reinician cada día (medianoche hora del Pacífico). Puedes ver el uso y los límites de tu proyecto en [AI Studio](https://aistudio.google.com/).
- Si quieres probar otro modelo, cambia `MODEL` (por ejemplo a `"gemini-3.8-flash"`), pero recuerda que los modelos 3.6+ ignoran `temperature`.
- Recuerda la regla del nivel gratuito: nada de datos personales ni confidenciales en los prompts.
- **Lectura relacionada:** Lanham (2024), capítulo sobre acciones y herramientas.


## Bonus: Implementación real con el SDK (Chat API)

Como vimos en la sección anterior, usar `generate_content` con llamadas automáticas a funciones genera un *warning* recomendando usar la API de Chat. En la práctica, cuando queremos que el SDK gestione todo el ciclo de vida del agente (el historial, las llamadas a herramientas y las respuestas) de forma limpia y óptima, utilizamos `client.chats.create`.

Al crear el chat, le pasamos la configuración del sistema (`system_instruction`) y las herramientas disponibles (`tools`). Luego, cada interacción se realiza mediante `chat.send_message(...)`, permitiendo que la librería gestione automáticamente todo el ciclo agéntico por debajo.

In [ ]:
# Creamos la configuración del chat incluyendo las herramientas y la instrucción de sistema
config_chat = types.GenerateContentConfig(
    system_instruction=INSTRUCCIONES,
    temperature=0,
    tools=[buscar_pedido, consultar_stock, estimar_despacho]
)

# Iniciamos un nuevo chat con la configuración establecida
chat = client.chats.create(model=MODEL, config=config_chat)

# Enviamos el mensaje al chat. El SDK resolverá las llamadas de función automáticamente sin arrojar warnings
respuesta = chat.send_message("¿Cuándo le llega a Clínica Norte su pedido P-1003?")

print(respuesta.text)

In [ ]:
respuesta = chat.send_message("¿Y qué contiene ese pedido?")
print(respuesta.text)